In [5]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from model import BrainAgeResNet
from training_state import load_checkpoint
from dataloader import load_mri

# 全部使用 /，避免 Windows 路徑中的 \t、\r 被解讀成跳脫字元
ROOT = Path("C:/Lab/test_retest_dataset")
TEST_CSV = Path("C:/Lab/models/ResNet_1015/test_datalist.csv")
VAL_CSV = Path("C:/Lab/models/ResNet_1015/val_datalist.csv")
CHECKPOINT = Path("C:/Lab/models/ResNet_1015/True/output/best.pt")
OUTPUT = Path("C:/Lab/models/ResNet_1015/True/output")
BATCH_SIZE = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUNS = [
    # "Test",
    "Val",
    # "MMRR",
    # "Maclaren",
    # "OAS1",
]
print("Device:", DEVICE)

Device: cuda


In [6]:
# Test 的 File_Path 是完整路徑；其他三組 Excel 的 rmi 是檔案名稱

def get_table(name):
    if name == "Test":
        df = pd.read_csv(TEST_CSV)
        df = df.rename(columns={"File_Path": "path"})
        # 原 CSV 的資料夾結構正確時直接沿用；若原根目錄不同則只替換根目錄
        df["path"] = df["path"].astype(str).map(
            lambda p: str(ROOT / "dataset_val_test" / p.replace("\\", "/").split("dataset_val_test/", 1)[1])
            if "dataset_val_test/" in p.replace("\\", "/") else p
        )
    elif name == "Val":
            df = pd.read_csv(VAL_CSV)
            df = df.rename(columns={"File_Path": "path"})
            # 原 CSV 的資料夾結構正確時直接沿用；若原根目錄不同則只替換根目錄
            df["path"] = df["path"].astype(str).map(
                lambda p: str(ROOT / "dataset_val_test" / p.replace("\\", "/").split("dataset_val_test/", 1)[1])
                if "dataset_val_test/" in p.replace("\\", "/") else p
            )
    else:
        folder = ROOT / name
        df = pd.read_excel(folder / f"{name}.xlsx")
        df = df.rename(columns={"Age": "AGE", "Sex (1=Male, 2=Female)": "SEX"})
        df["path"] = df["rmi"].astype(str).map(
            lambda x: str(folder / "ndf" / "rmi" / (x if x.endswith((".nii", ".nii.gz")) else x + ".nii"))
        )
    for p in df["path"]:
        if not Path(p).is_file():
            raise FileNotFoundError(f"{name}: 找不到影像 {p}")
    df["sex_model"] = df["SEX"].map({1: 0.0, 2: 1.0})
    df["scanner_model"] = df["Scanner"].map({1.5: 0.0, 3.0: 1.0})
    if df[["sex_model", "scanner_model", "AGE"]].isna().any().any():
        raise ValueError(f"{name}: AGE、SEX 或 Scanner 內容不符合預期")
    return df

class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        return (load_mri(row["path"]),
                torch.tensor(row["sex_model"], dtype=torch.float32),
                torch.tensor(row["scanner_model"], dtype=torch.float32),
                float(row["AGE"]), str(row["Subject_ID"]))

In [7]:
# Checkpoint 是訓練儲存的模型檔，best.pt 包含最佳權重與模型設定
checkpoint = load_checkpoint(CHECKPOINT)
if checkpoint["training_config"]["registration"] != "rigid":
    raise ValueError("best.pt 並非 rigid 模型")
model = BrainAgeResNet(**checkpoint["model_config"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model ready; best epoch:", checkpoint["epoch"])

Model ready; best epoch: 224


In [8]:
OUTPUT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

with torch.inference_mode():
    for name in RUNS:
        df = get_table(name)
        loader = DataLoader(TestDataset(df), batch_size=BATCH_SIZE,
                            shuffle=False, drop_last=False, num_workers=0)
        results = []
        for image, sex, scanner, ages, subjects in tqdm(loader, desc=name):
            predicted = model(image.to(DEVICE),
                              sex.to(DEVICE).unsqueeze(1),
                              scanner.to(DEVICE).unsqueeze(1)).squeeze(1).cpu().numpy()
            results.extend(zip(subjects, ages.tolist(), predicted.tolist()))

        output_df = pd.DataFrame(results, columns=["Subject ID", "True Age", "Predicted Age"])
        filename = OUTPUT / f"ResNet_{name}_{timestamp}.xlsx"
        output_df.to_excel(filename, sheet_name="Sheet1", index=False)
        error = output_df["Predicted Age"] - output_df["True Age"]
        print(f"{name}: N={len(output_df)}, MAE={error.abs().mean():.4f}, "
              f"RMSE={np.sqrt((error**2).mean()):.4f} | {filename}")

Val: 100%|██████████| 464/464 [02:10<00:00,  3.57it/s]

Val: N=464, MAE=3.5127, RMSE=4.6686 | C:\Lab\models\ResNet_1015\True\output\ResNet_Val_20261027_144149.xlsx


In [13]:

from pathlib import Path
from datetime import datetime
import pandas as pd
import re

output_dir = Path("./output")
files = sorted(output_dir.glob("ResNet_*_T*_*.xlsx"))

sheets = {}

for file in files:
    match = re.fullmatch(r"ResNet_(.+)_T(\d+)_(\d{8}_\d{6})\.xlsx", file.name)
    if match is None:
        continue

    dataset, batch_size, _ = match.groups()
    sheet_name = f"{dataset}_T{batch_size}"

    if sheet_name in sheets:
        raise ValueError(f"重複的 Sheet 名稱：{sheet_name}，請先移除舊版本檔案")

    sheets[sheet_name] = pd.read_excel(file)

if not sheets:
    raise FileNotFoundError("找不到符合命名規則的 ResNet Excel 檔案")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = output_dir / f"ResNet_{timestamp}.xlsx"

with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"合併完成：{save_path}")
print("Sheets：", list(sheets))

合併完成：output\ResNet_20261027_132913.xlsx
Sheets： ['Maclaren_T1', 'Maclaren_T4', 'MMRR_T1', 'MMRR_T4', 'OAS1_T1', 'OAS1_T4', 'Test_T1', 'Test_T4']
